In [1]:
import os, json, zipfile, math, copy, sys, time, traceback, random
import numpy as np
from pathlib import Path

import onnx
import onnx.helper as oh
import onnx.numpy_helper as onh
from onnx import TensorProto

try:
    import onnxruntime as ort
except ImportError:
    os.system("pip install onnxruntime -q")
    import onnxruntime as ort

# ============================================================
# CONFIG
# ============================================================
TASK_DIR = Path("/kaggle/input/competitions/neurogolf-2026")
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
C, H, W = 10, 30, 30
HW = H * W
CHW = C * H * W
MAX_ARC_GEN_VALIDATE = 50  # Validate on up to this many arc-gen pairs

# ============================================================
# DATA UTILS
# ============================================================
def load_task(path):
    with open(path) as f:
        return json.load(f)

def grid_to_tensor(grid):
    g = np.array(grid, dtype=np.int32)
    h, w = g.shape
    t = np.zeros((1, C, H, W), dtype=np.float32)
    if h > H or w > W:
        return t
    for v in range(C):
        t[0, v, :h, :w] = (g == v).astype(np.float32)
    return t

def tensor_to_grid(t, out_h, out_w):
    arr = t[0]
    out_h = min(out_h, H)
    out_w = min(out_w, W)
    slice_ = arr[:, :out_h, :out_w]
    max_vals = slice_.max(axis=0)
    argmax = slice_.argmax(axis=0)
    grid = np.where(max_vals < 0.5, 0, argmax).astype(int)
    return grid.tolist()

def shapes_match(p):
    return np.array(p["input"]).shape == np.array(p["output"]).shape

# ============================================================
# MODEL CHECK + PROPER COST (WITH MACs)
# ============================================================
def check_model_correct(model, pairs):
    try:
        buf = model.SerializeToString()
        sess = ort.InferenceSession(buf, providers=["CPUExecutionProvider"])
        for p in pairs:
            inp = grid_to_tensor(p["input"])
            out_g = np.array(p["output"])
            oh_, ow_ = out_g.shape
            pred = sess.run(None, {"input": inp})[0]
            pred_grid = tensor_to_grid(pred, oh_, ow_)
            if pred_grid != p["output"]:
                return False
        return True
    except Exception:
        return False

def estimate_model_cost(model):
    """Cost = params + memory_bytes + MACs.
    Critical: V2 was missing MACs! This changes strategy ranking."""
    try:
        total_params = 0
        total_bytes = 0

        # Collect all initializers and constants with their arrays
        tensors = {}  # name → array
        for init in model.graph.initializer:
            arr = onh.to_array(init)
            tensors[init.name] = arr
            total_params += arr.size
            total_bytes += arr.nbytes
        for node in model.graph.node:
            if node.op_type == "Constant":
                for attr in node.attribute:
                    if attr.t and (attr.t.raw_data or list(attr.t.float_data)
                                   or list(attr.t.int64_data) or list(attr.t.int32_data)):
                        arr = onh.to_array(attr.t)
                        if node.output:
                            tensors[node.output[0]] = arr
                        total_params += arr.size
                        total_bytes += arr.nbytes

        # Estimate MACs
        total_macs = 0
        for node in model.graph.node:
            if node.op_type == "Conv":
                if len(node.input) >= 2 and node.input[1] in tensors:
                    w = tensors[node.input[1]]
                    if w.ndim == 4:
                        c_out, c_in, kh, kw = w.shape
                        # Output spatial = input spatial (we use 'same' padding)
                        total_macs += c_out * c_in * kh * kw * H * W
            elif node.op_type == "Gemm":
                if len(node.input) >= 2 and node.input[1] in tensors:
                    w = tensors[node.input[1]]
                    if w.ndim == 2:
                        m_dim = 1  # batch
                        total_macs += m_dim * w.shape[0] * w.shape[1]
            elif node.op_type in ("Mul", "Add", "Max", "Relu"):
                # Element-wise ops — count each elem as 1 "op"
                # Use output size = HW*C for our graphs
                total_macs += CHW
            # Gather, Reshape, Identity: 0 MACs

        return total_params + total_bytes + total_macs
    except Exception as e:
        return float('inf')

# ============================================================
# ONNX BUILDERS (minimized for cost)
# ============================================================

def make_identity_onnx():
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    node = oh.make_node("Identity", ["input"], ["output"])
    graph = oh.make_graph([node], "identity", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_const_onnx(const_output):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    const_val = const_output.astype(np.float32)
    const_tensor = onh.from_array(const_val, name="const_out")
    const_node = oh.make_node("Constant", [], ["const_out"], value=const_tensor)
    zero_tensor = onh.from_array(np.zeros((1, C, H, W), dtype=np.float32), name="zero_val")
    zero_node = oh.make_node("Constant", [], ["zero_val"], value=zero_tensor)
    mul_node = oh.make_node("Mul", ["input", "zero_val"], ["zeroed"])
    add_node = oh.make_node("Add", ["zeroed", "const_out"], ["output"])
    graph = oh.make_graph([const_node, zero_node, mul_node, add_node], "const_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def _compact_mask(mask):
    """Reduce [1, C, H, W] mask to [1, 1, H, W] when all channels match."""
    if mask.ndim == 4 and mask.shape[0] == 1 and mask.shape[1] == C:
        if np.all(mask[0, 0:1] == mask[0, :]):
            return mask[:, 0:1].copy()
    return mask

def make_gather_onnx(gather_indices, mask=None):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    shape_chw = onh.from_array(np.array([1, C, HW], dtype=np.int64), name="shape_chw")
    shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
    inp_reshape = oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"])

    max_idx = int(gather_indices.max()) if gather_indices.size > 0 else 0
    if max_idx < 32768:
        gi_tensor = onh.from_array(gather_indices.astype(np.int32), name="gather_idx")
    else:
        gi_tensor = onh.from_array(gather_indices.astype(np.int64), name="gather_idx")
    gi_node = oh.make_node("Constant", [], ["gather_idx"], value=gi_tensor)
    gather_node = oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2)

    out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
    out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)

    if mask is not None:
        reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["gathered_4d"])
        mask = _compact_mask(mask)
        mask_tensor = onh.from_array(mask.astype(np.float32), name="mask")
        mask_node = oh.make_node("Constant", [], ["mask"], value=mask_tensor)
        mul_node = oh.make_node("Mul", ["gathered_4d", "mask"], ["output"])
        nodes = [shape_chw_node, inp_reshape, gi_node, gather_node,
                 out_shape_node, reshape_back, mask_node, mul_node]
    else:
        reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["output"])
        nodes = [shape_chw_node, inp_reshape, gi_node, gather_node, out_shape_node, reshape_back]

    graph = oh.make_graph(nodes, "gather_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_conv1x1_onnx(weight):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    w_tensor = onh.from_array(weight.astype(np.float32), name="conv_weight")
    w_node = oh.make_node("Constant", [], ["conv_weight"], value=w_tensor)
    conv_node = oh.make_node("Conv", ["input", "conv_weight"], ["output"],
                             kernel_shape=[1, 1], pads=[0, 0, 0, 0])
    graph = oh.make_graph([w_node, conv_node], "conv1x1", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_channel_gather_onnx(gather_ch):
    """Color map via channel gather. Only works for permutations (bijective maps).
    Cost: ~50 vs 90500 for 1x1 conv. Score: ~20.5 vs 13.6. MASSIVE win.
    """
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    gi_tensor = onh.from_array(gather_ch.astype(np.int32), name="gi")
    gi_node = oh.make_node("Constant", [], ["gi"], value=gi_tensor)
    gather = oh.make_node("Gather", ["input", "gi"], ["output"], axis=1)
    graph = oh.make_graph([gi_node, gather], "ch_gather", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_spatial_then_channel_gather_onnx(spatial_idx, channel_idx):
    """Two gathers: spatial then channel. Used for composite transforms."""
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])

    shape_chw = onh.from_array(np.array([1, C, HW], dtype=np.int64), name="shape_chw")
    shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
    inp_reshape = oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"])

    si = onh.from_array(spatial_idx.astype(np.int32), name="si")
    si_node = oh.make_node("Constant", [], ["si"], value=si)
    spatial_gather = oh.make_node("Gather", ["inp_chw", "si"], ["spatial_out"], axis=2)

    out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
    out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)
    reshape_back = oh.make_node("Reshape", ["spatial_out", "out_shape"], ["spatial_4d"])

    ci = onh.from_array(channel_idx.astype(np.int32), name="ci")
    ci_node = oh.make_node("Constant", [], ["ci"], value=ci)
    channel_gather = oh.make_node("Gather", ["spatial_4d", "ci"], ["output"], axis=1)

    graph = oh.make_graph([shape_chw_node, inp_reshape, si_node, spatial_gather,
                           out_shape_node, reshape_back, ci_node, channel_gather],
                          "spatial_ch_gather", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_conv_onnx(weight, bias=None, kernel_size=3):
    pad = kernel_size // 2
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    w_tensor = onh.from_array(weight.astype(np.float32), name="conv_weight")
    w_node = oh.make_node("Constant", [], ["conv_weight"], value=w_tensor)
    if bias is not None:
        b_tensor = onh.from_array(bias.astype(np.float32), name="conv_bias")
        b_node = oh.make_node("Constant", [], ["conv_bias"], value=b_tensor)
        conv_node = oh.make_node("Conv", ["input", "conv_weight", "conv_bias"], ["output"],
                                 kernel_shape=[kernel_size, kernel_size],
                                 pads=[pad, pad, pad, pad])
        nodes = [w_node, b_node, conv_node]
    else:
        conv_node = oh.make_node("Conv", ["input", "conv_weight"], ["output"],
                                 kernel_shape=[kernel_size, kernel_size],
                                 pads=[pad, pad, pad, pad])
        nodes = [w_node, conv_node]
    graph = oh.make_graph(nodes, "conv_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_gather_then_conv1x1_onnx(gather_indices, conv_weight, mask=None):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])

    shape_chw = onh.from_array(np.array([1, C, HW], dtype=np.int64), name="shape_chw")
    shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
    inp_reshape = oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"])
    gi_tensor = onh.from_array(gather_indices.astype(np.int32), name="gather_idx")
    gi_node = oh.make_node("Constant", [], ["gather_idx"], value=gi_tensor)
    gather_node = oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2)
    out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
    out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)
    reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["gathered_4d"])

    w_tensor = onh.from_array(conv_weight.astype(np.float32), name="conv_weight")
    w_node = oh.make_node("Constant", [], ["conv_weight"], value=w_tensor)

    if mask is not None:
        mask = _compact_mask(mask)
        mask_tensor = onh.from_array(mask.astype(np.float32), name="mask")
        mask_node = oh.make_node("Constant", [], ["mask"], value=mask_tensor)
        mask_mul = oh.make_node("Mul", ["gathered_4d", "mask"], ["masked"])
        conv_node = oh.make_node("Conv", ["masked", "conv_weight"], ["output"],
                                 kernel_shape=[1, 1], pads=[0, 0, 0, 0])
        nodes = [shape_chw_node, inp_reshape, gi_node, gather_node,
                 out_shape_node, reshape_back, mask_node, mask_mul, w_node, conv_node]
    else:
        conv_node = oh.make_node("Conv", ["gathered_4d", "conv_weight"], ["output"],
                                 kernel_shape=[1, 1], pads=[0, 0, 0, 0])
        nodes = [shape_chw_node, inp_reshape, gi_node, gather_node,
                 out_shape_node, reshape_back, w_node, conv_node]

    graph = oh.make_graph(nodes, "gather_conv1x1", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_symmetry_completion_onnx(gather_index_list, mask=None):
    """MAX over multiple Gather ops, with channel-0 (bg) suppressed."""
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])

    ch_mask = np.ones((1, C, 1, 1), dtype=np.float32); ch_mask[0, 0, 0, 0] = 0.0
    ch_mask_tensor = onh.from_array(ch_mask, name="ch_mask")
    ch_mask_node = oh.make_node("Constant", [], ["ch_mask"], value=ch_mask_tensor)
    nobg_mul = oh.make_node("Mul", ["input", "ch_mask"], ["input_nobg"])

    shape_chw = onh.from_array(np.array([1, C, HW], dtype=np.int64), name="shape_chw")
    shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
    inp_reshape = oh.make_node("Reshape", ["input_nobg", "shape_chw"], ["inp_chw"])
    out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
    out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)

    nodes = [ch_mask_node, nobg_mul, shape_chw_node, inp_reshape, out_shape_node]
    reshape_names = []
    for i, gi in enumerate(gather_index_list):
        gi_name = f"gather_idx_{i}"
        gi_tensor = onh.from_array(gi.astype(np.int32), name=gi_name)
        gi_node = oh.make_node("Constant", [], [gi_name], value=gi_tensor)
        gname = f"gathered_{i}"
        gather_node = oh.make_node("Gather", ["inp_chw", gi_name], [gname], axis=2)
        rname = f"reshaped_{i}"
        reshape_node = oh.make_node("Reshape", [gname, "out_shape"], [rname])
        nodes.extend([gi_node, gather_node, reshape_node])
        reshape_names.append(rname)

    if len(reshape_names) == 1:
        final = reshape_names[0]
    else:
        prev = reshape_names[0]
        for i in range(1, len(reshape_names)):
            out_name = "output" if (i == len(reshape_names) - 1 and mask is None) else f"max_{i}"
            max_node = oh.make_node("Max", [prev, reshape_names[i]], [out_name])
            nodes.append(max_node)
            prev = out_name
        final = prev

    if mask is not None:
        mask = _compact_mask(mask)
        mask_tensor = onh.from_array(mask.astype(np.float32), name="mask")
        mask_node = oh.make_node("Constant", [], ["mask"], value=mask_tensor)
        mul_node = oh.make_node("Mul", [final, "mask"], ["output"])
        nodes.extend([mask_node, mul_node])
    elif len(reshape_names) == 1:
        nodes.append(oh.make_node("Identity", [final], ["output"]))

    graph = oh.make_graph(nodes, "sym_complete", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

# ============================================================
# ANALYSIS
# ============================================================
def analyze_transformation(pairs):
    info = {}
    in_sizes = [np.array(p["input"]).shape for p in pairs]
    out_sizes = [np.array(p["output"]).shape for p in pairs]
    info["in_sizes"] = in_sizes
    info["out_sizes"] = out_sizes
    info["same_size"] = all(i == o for i, o in zip(in_sizes, out_sizes))
    info["const_output"] = len(set(map(lambda p: json.dumps(p["output"]), pairs))) == 1
    info["all_same_in_size"] = len(set(in_sizes)) == 1
    info["all_same_out_size"] = len(set(out_sizes)) == 1

    if info["same_size"]:
        diffs = [np.sum(np.array(p["input"]) != np.array(p["output"])) for p in pairs]
        info["identity"] = all(d == 0 for d in diffs)
    else:
        info["identity"] = False

    color_maps = []
    for p in pairs:
        ig = np.array(p["input"]).flatten()
        og = np.array(p["output"]).flatten()
        if ig.shape == og.shape:
            cmap = {}; valid = True
            for a, b in zip(ig, og):
                if a in cmap and cmap[a] != b:
                    valid = False; break
                cmap[a] = b
            if valid:
                color_maps.append(cmap)
    if color_maps and len(color_maps) == len(pairs):
        common = color_maps[0]
        for cm in color_maps[1:]:
            if cm != common: common = None; break
        info["color_map"] = common
    else:
        info["color_map"] = None
    return info

# ============================================================
# DETECTORS
# ============================================================

def detect_rotation(pairs):
    for angle in [90, 180, 270]:
        k = angle // 90
        valid = True
        for p in pairs:
            ig = np.array(p["input"]); og = np.array(p["output"])
            rotated = np.rot90(ig, k)
            if rotated.shape != og.shape or not np.array_equal(rotated, og):
                valid = False; break
        if valid: return angle
    return None

def detect_flip(pairs):
    for axis_val, name in [(0, "vertical"), (1, "horizontal"), (-1, "both")]:
        valid = True
        for p in pairs:
            ig = np.array(p["input"]); og = np.array(p["output"])
            if axis_val == -1:
                flipped = np.flip(np.flip(ig, 0), 1)
            else:
                flipped = np.flip(ig, axis=axis_val)
            if flipped.shape != og.shape or not np.array_equal(flipped, og):
                valid = False; break
        if valid: return name
    return None

def detect_transpose(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if ig.T.shape != og.shape or not np.array_equal(ig.T, og):
            return False
    return True

def detect_anti_transpose(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        anti = np.flip(np.flip(ig.T, 0), 1)
        if anti.shape != og.shape or not np.array_equal(anti, og):
            return False
    return True

def detect_crop(pairs):
    results = []
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ > ih or ow_ > iw: return None
        found = False
        for r in range(ih - oh_ + 1):
            for c in range(iw - ow_ + 1):
                if np.array_equal(ig[r:r+oh_, c:c+ow_], og):
                    results.append((r, c, oh_, ow_))
                    found = True; break
            if found: break
        if not found: return None
    if len(results) == len(pairs):
        offsets = [(r[0], r[1]) for r in results]
        if len(set(offsets)) == 1:
            return (offsets[0][0], offsets[0][1], results[0][2], results[0][3])
    return None

def detect_tile(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ % ih != 0 or ow_ % iw != 0: return None
        tr = oh_ // ih; tc = ow_ // iw
        if not np.array_equal(np.tile(ig, (tr, tc)), og): return None
    ig0 = np.array(pairs[0]["input"]); og0 = np.array(pairs[0]["output"])
    return (og0.shape[0] // ig0.shape[0], og0.shape[1] // ig0.shape[1])

def detect_scale(pairs):
    for factor in [2, 3, 4, 5]:
        valid = True
        for p in pairs:
            ig = np.array(p["input"]); og = np.array(p["output"])
            ih, iw = ig.shape; oh_, ow_ = og.shape
            if oh_ != ih * factor or ow_ != iw * factor:
                valid = False; break
            if not np.array_equal(np.repeat(np.repeat(ig, factor, 0), factor, 1), og):
                valid = False; break
        if valid: return factor
    return None

def detect_upscale(pairs):
    for yf, xf in [(1,2),(2,1),(1,3),(3,1),(2,3),(3,2),(1,4),(4,1)]:
        valid = True
        for p in pairs:
            ig = np.array(p["input"]); og = np.array(p["output"])
            ih, iw = ig.shape; oh_, ow_ = og.shape
            if oh_ != ih * yf or ow_ != iw * xf:
                valid = False; break
            if not np.array_equal(np.repeat(np.repeat(ig, yf, 0), xf, 1), og):
                valid = False; break
        if valid: return (yf, xf)
    return None

def detect_color_replace(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    replacements = {}
    for p in pairs:
        ig = np.array(p["input"]).flatten()
        og = np.array(p["output"]).flatten()
        for a, b in zip(ig, og):
            if a != b:
                if a in replacements and replacements[a] != b:
                    return None
                replacements[a] = b
    if not replacements: return None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        expected = np.copy(ig)
        for src, dst in replacements.items():
            expected[ig == src] = dst
        if not np.array_equal(expected, og): return None
    return replacements

def detect_color_swap(pairs):
    """Only accept swap if we have evidence of BOTH directions. E.g., we need
    at least one pair where 'a' appears in input (becomes 'b' in output),
    AND at least one pair where 'b' appears in input (becomes 'a' in output).
    Otherwise it's ambiguous with a simple replace."""
    if not all(shapes_match(p) for p in pairs): return None
    swap_pairs = set()
    for p in pairs:
        ig = np.array(p["input"]).flatten(); og = np.array(p["output"]).flatten()
        for a, b in zip(ig, og):
            if a != b:
                swap_pairs.add((min(a,b), max(a,b)))
    if len(swap_pairs) != 1: return None
    a, b = list(swap_pairs)[0]

    saw_a_to_b = False
    saw_b_to_a = False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        expected = np.copy(ig)
        expected[ig == a] = b; expected[ig == b] = a
        if not np.array_equal(expected, og): return None
        # Evidence requires a pixel with input==a mapping to output==b AND vice versa
        if np.any((ig == a) & (og == b)): saw_a_to_b = True
        if np.any((ig == b) & (og == a)): saw_b_to_a = True
    if not (saw_a_to_b and saw_b_to_a):
        return None  # Ambiguous — could be a replace, not a swap
    return (a, b)

def detect_pixel_permutation(pairs):
    """CONSERVATIVE: requires consistent mapping across ALL pairs."""
    if not all(shapes_match(p) for p in pairs): return None
    if len(set(np.array(p["input"]).shape for p in pairs)) != 1: return None
    ih, iw = np.array(pairs[0]["input"]).shape
    if ih > H or iw > W: return None

    # For each destination position, find which source matches across ALL pairs
    n_pairs = len(pairs)
    all_inputs = [np.array(p["input"]).flatten() for p in pairs]
    all_outputs = [np.array(p["output"]).flatten() for p in pairs]

    grid_size = ih * iw
    assignment = {}
    for dst_idx in range(grid_size):
        # What are the target values at dst_idx across pairs?
        targets = [all_outputs[pi][dst_idx] for pi in range(n_pairs)]
        # Find src_idx such that all_inputs[pi][src_idx] == targets[pi] for all pi
        candidates = None
        for pi in range(n_pairs):
            match = set(np.where(all_inputs[pi] == targets[pi])[0].tolist())
            candidates = match if candidates is None else candidates & match
            if not candidates: return None
        if len(candidates) != 1: return None
        assignment[dst_idx] = candidates.pop()

    gather_indices = np.arange(HW, dtype=np.int64)
    for dst_idx, src_idx in assignment.items():
        dst_r, dst_c = divmod(dst_idx, iw)
        src_r, src_c = divmod(src_idx, iw)
        gather_indices[dst_r * W + dst_c] = src_r * W + src_c

    # Final verification
    for p in pairs:
        ig = np.array(p["input"])
        padded = np.zeros((H, W), dtype=np.int32)
        padded[:ih, :iw] = ig
        flat = padded.flatten()
        pred = flat[gather_indices].reshape(H, W)[:ih, :iw]
        if not np.array_equal(pred, np.array(p["output"])):
            return None
    return gather_indices

def detect_mirror_h_concat(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ != ih or ow_ != 2*iw: return False
        if not np.array_equal(np.concatenate([ig, np.flip(ig, 1)], 1), og): return False
    return True

def detect_mirror_v_concat(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ != 2*ih or ow_ != iw: return False
        if not np.array_equal(np.concatenate([ig, np.flip(ig, 0)], 0), og): return False
    return True

def detect_self_concat_h(pairs):
    """Output = [input, input]."""
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ != ih or ow_ != 2*iw: return False
        if not np.array_equal(np.concatenate([ig, ig], 1), og): return False
    return True

def detect_self_concat_v(pairs):
    """Output = [input; input]."""
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ != 2*ih or ow_ != iw: return False
        if not np.array_equal(np.concatenate([ig, ig], 0), og): return False
    return True

def detect_quad_mirror(pairs):
    patterns = []
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ != 2*ih or ow_ != 2*iw: return None
        exp1 = np.block([[ig, np.flip(ig,1)], [np.flip(ig,0), np.flip(np.flip(ig,0),1)]])
        if np.array_equal(exp1, og): patterns.append("tl"); continue
        exp2 = np.block([[np.flip(np.flip(ig,0),1), np.flip(ig,0)], [np.flip(ig,1), ig]])
        if np.array_equal(exp2, og): patterns.append("br"); continue
        return None
    if patterns and len(set(patterns)) == 1:
        return patterns[0]
    return None

def detect_nonzero_recolor(pairs):
    """Detect: all non-bg pixels get same color; bg stays bg."""
    if not all(shapes_match(p) for p in pairs): return None
    target_color = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        # bg must stay bg
        if not np.all(og[ig == 0] == 0): return None
        # all non-bg pixels → single color
        nonbg = og[ig != 0]
        if nonbg.size == 0: continue
        unique = np.unique(nonbg)
        if len(unique) != 1: return None
        if target_color is None:
            target_color = unique[0]
        elif target_color != unique[0]:
            return None
    return target_color

def detect_bg_recolor(pairs):
    """Detect: bg pixels (0) → other color, non-bg stay same."""
    if not all(shapes_match(p) for p in pairs): return None
    target = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        # non-bg must stay same
        if not np.array_equal(og[ig != 0], ig[ig != 0]): return None
        # bg → target
        bg_out = og[ig == 0]
        if bg_out.size == 0: continue
        u = np.unique(bg_out)
        if len(u) != 1: return None
        if target is None:
            target = u[0]
        elif target != u[0]:
            return None
    return target

def detect_h_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        expected = np.where(ig != 0, ig, np.flip(ig, 1))
        if not np.array_equal(expected, og): return False
    return True

def detect_v_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        expected = np.where(ig != 0, ig, np.flip(ig, 0))
        if not np.array_equal(expected, og): return False
    return True

def detect_both_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        m_h = np.flip(ig, 1); m_v = np.flip(ig, 0); m_b = np.flip(np.flip(ig,0),1)
        stack = np.stack([ig, m_h, m_v, m_b], 0)
        expected = np.where(ig != 0, ig, stack.max(0))
        if not np.array_equal(expected, og): return False
    return True

def detect_diag_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if ig.shape[0] != ig.shape[1]: return False
        expected = np.where(ig != 0, ig, ig.T)
        if not np.array_equal(expected, og): return False
    return True

def detect_all4_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if ig.shape[0] != ig.shape[1]: return False
        m_h = np.flip(ig, 1); m_v = np.flip(ig, 0); m_b = np.flip(np.flip(ig,0),1); m_t = ig.T
        stack = np.stack([ig, m_h, m_v, m_b, m_t], 0)
        expected = np.where(ig != 0, ig, stack.max(0))
        if not np.array_equal(expected, og): return False
    return True

def detect_grid_of_grids_fixed(pairs):
    """Output is always the same block position."""
    block_selections = []
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ >= ih or ow_ >= iw: return None
        if ih % oh_ != 0 or iw % ow_ != 0: return None
        blocks_r = ih // oh_; blocks_c = iw // ow_
        found = None
        for br in range(blocks_r):
            for bc in range(blocks_c):
                if np.array_equal(ig[br*oh_:(br+1)*oh_, bc*ow_:(bc+1)*ow_], og):
                    found = (br, bc); break
            if found: break
        if found is None: return None
        block_selections.append(found + (oh_, ow_))
    if len(set(block_selections)) == 1:
        br, bc, oh_, ow_ = block_selections[0]
        return (br, bc, oh_, ow_)
    return None

# ============================================================
# GATHER INDEX BUILDERS
# ============================================================

SAFE_PAD_IDX = HW - 1  # Gather here for "don't care" regions; input[H-1,W-1] is padded (all-zero in one-hot)

def build_rotation_gather(angle, ih, iw):
    """Build gather for np.rot90(input, k). Unused positions point to safe padding."""
    k = angle // 90
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    if k == 1:
        for r in range(iw):
            for c in range(ih):
                idx[r*W + c] = c*W + (iw - 1 - r)
    elif k == 2:
        for r in range(ih):
            for c in range(iw):
                idx[r*W + c] = (ih-1-r)*W + (iw-1-c)
    elif k == 3:
        for r in range(iw):
            for c in range(ih):
                idx[r*W + c] = (ih-1-c)*W + r
    return idx

def build_flip_gather(direction, ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            if direction == "vertical":
                idx[r*W+c] = (ih-1-r)*W + c
            elif direction == "horizontal":
                idx[r*W+c] = r*W + (iw-1-c)
            else:
                idx[r*W+c] = (ih-1-r)*W + (iw-1-c)
    return idx

def build_transpose_gather(ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(iw):
        for c in range(ih):
            idx[r*W+c] = c*W + r
    return idx

def build_anti_transpose_gather(ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(iw):
        for c in range(ih):
            sr = ih - 1 - c; sc = iw - 1 - r
            if 0 <= sr < H and 0 <= sc < W:
                idx[r*W+c] = sr*W + sc
    return idx

def build_crop_gather(r0, c0, oh_, ow_):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(oh_):
        for c in range(ow_):
            sr = r0 + r; sc = c0 + c
            if sr < H and sc < W:
                idx[r*W+c] = sr*W + sc
    return idx

def build_tile_gather(tr, tc, ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(min(H, ih*tr)):
        for c in range(min(W, iw*tc)):
            idx[r*W+c] = (r % ih) * W + (c % iw)
    return idx

def build_scale_gather(factor, ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(min(H, ih*factor)):
        for c in range(min(W, iw*factor)):
            idx[r*W+c] = (r//factor)*W + (c//factor)
    return idx

def build_nonuniform_scale_gather(yf, xf, ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(min(H, ih*yf)):
        for c in range(min(W, iw*xf)):
            idx[r*W+c] = (r//yf)*W + (c//xf)
    return idx

def build_mirror_h_concat_gather(ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(2*iw):
            if c < W:
                src_c = c if c < iw else (2*iw-1-c)
                idx[r*W+c] = r*W + src_c
    return idx

def build_mirror_v_concat_gather(ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(2*ih):
        for c in range(iw):
            if r < H:
                src_r = r if r < ih else (2*ih-1-r)
                idx[r*W+c] = src_r*W + c
    return idx

def build_self_concat_h_gather(ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(2*iw):
            if c < W:
                idx[r*W+c] = r*W + (c % iw)
    return idx

def build_self_concat_v_gather(ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(2*ih):
        for c in range(iw):
            if r < H:
                idx[r*W+c] = (r % ih)*W + c
    return idx

def build_quad_mirror_gather(ih, iw, corner="tl"):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    if corner == "tl":
        for r in range(min(H, 2*ih)):
            for c in range(min(W, 2*iw)):
                sr = r if r < ih else (2*ih-1-r)
                sc = c if c < iw else (2*iw-1-c)
                idx[r*W+c] = sr*W + sc
    else:
        for r in range(min(H, 2*ih)):
            for c in range(min(W, 2*iw)):
                sr = (r - ih) if r >= ih else (ih - 1 - r)
                sc = (c - iw) if c >= iw else (iw - 1 - c)
                idx[r*W+c] = sr*W + sc
    return idx

def build_h_sym_complete_gathers(ih, iw):
    idx_id = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    idx_m = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            idx_id[r*W+c] = r*W + c
            idx_m[r*W+c] = r*W + (iw-1-c)
    return [idx_id, idx_m]

def build_v_sym_complete_gathers(ih, iw):
    idx_id = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    idx_m = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            idx_id[r*W+c] = r*W + c
            idx_m[r*W+c] = (ih-1-r)*W + c
    return [idx_id, idx_m]

def build_both_sym_complete_gathers(ih, iw):
    idxs = []
    for t in range(4):
        idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
        for r in range(ih):
            for c in range(iw):
                if t == 0: sr, sc = r, c
                elif t == 1: sr, sc = r, iw-1-c
                elif t == 2: sr, sc = ih-1-r, c
                else: sr, sc = ih-1-r, iw-1-c
                idx[r*W+c] = sr*W + sc
        idxs.append(idx)
    return idxs

def build_diag_sym_complete_gathers(ih, iw):
    if ih != iw: return None
    idx_id = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    idx_m = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            idx_id[r*W+c] = r*W + c
            idx_m[r*W+c] = c*W + r
    return [idx_id, idx_m]

def build_all4_sym_complete_gathers(ih, iw):
    if ih != iw: return None
    idxs = build_both_sym_complete_gathers(ih, iw)
    idx_t = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            idx_t[r*W+c] = c*W + r
    idxs.append(idx_t)
    return idxs

# ============================================================
# WEIGHT BUILDERS
# ============================================================
def can_use_channel_gather_for_cmap(cmap, train_pairs):
    """Check if channel-gather is SAFE for this color map on this training data.

    Safe conditions:
    - Bijective (swap/cycle): every src is also some dst.
    - OR: for each non-identity src→dst, no training input contains value `dst`
      (so there's no ambiguity between "replace" and "swap").

    Returns True if safe, False if ambiguous (should use conv1x1 instead).
    """
    if cmap is None: return False
    changed = {s: d for s, d in cmap.items() if s != d}
    if not changed: return False

    # Case 1: fully bijective - always safe
    if set(changed.keys()) == set(changed.values()):
        return True

    # Case 2: non-bijective. We could use channel-gather IF the destinations
    # don't appear in any input (so our swap-like behavior is equivalent to replace).
    # But this is fragile - private data might contain the destination color.
    # Return False to prefer conv1x1.
    return False

def build_channel_gather_indices(cmap):
    """For a bijective color_map, build channel indices such that
    output_channel[dst] = input_channel[src] for each mapping."""
    gather_ch = np.arange(C, dtype=np.int32)
    for src, dst in cmap.items():
        if src != dst:
            gather_ch[dst] = src
    return gather_ch

def build_color_map_weight(color_map):
    weight = np.zeros((C, C, 1, 1), dtype=np.float32)
    for src, dst in color_map.items():
        if 0 <= src < C and 0 <= dst < C:
            weight[dst, src, 0, 0] = 1.0
    for ch in range(C):
        if ch not in color_map:
            weight[ch, ch, 0, 0] = 1.0
    return weight

def build_nonzero_recolor_weight(target_color):
    """1x1 conv: all non-bg channels → target; bg → bg."""
    weight = np.zeros((C, C, 1, 1), dtype=np.float32)
    weight[0, 0, 0, 0] = 1.0  # bg → bg
    for src in range(1, C):
        weight[target_color, src, 0, 0] = 1.0
    return weight

def build_bg_recolor_weight(target_color):
    """1x1 conv: bg → target; non-bg stay."""
    weight = np.zeros((C, C, 1, 1), dtype=np.float32)
    weight[target_color, 0, 0, 0] = 1.0
    for src in range(1, C):
        weight[src, src, 0, 0] = 1.0
    return weight

# ============================================================
# LEARNED CONV with data augmentation
# ============================================================

def augment_pairs(pairs):
    """Create rotation/flip augmented training pairs. Only safe when the
    transformation is rotation/flip equivariant — a strong assumption.
    We return just originals here; we don't know in general.
    """
    return pairs  # Conservative: no augmentation by default (breaks non-equivariant tasks)

def try_learned_conv_fast(pairs, all_pairs, kernel_size=1, max_steps=1500, lr=0.02):
    try:
        import torch, torch.nn as nn
    except ImportError:
        return None
    if not all(shapes_match(p) for p in pairs): return None

    inp = torch.tensor(np.stack([grid_to_tensor(p["input"])[0] for p in pairs]), dtype=torch.float32)
    out = torch.tensor(np.stack([grid_to_tensor(p["output"])[0] for p in pairs]), dtype=torch.float32)

    pad = kernel_size // 2
    conv = nn.Conv2d(C, C, kernel_size=kernel_size, padding=pad, bias=False)
    nn.init.zeros_(conv.weight)
    for i in range(C):
        conv.weight.data[i, i, pad, pad] = 1.0  # near-identity init

    optimizer = torch.optim.Adam(conv.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)
    best_loss = float("inf"); best_state = None; plateau = 0

    for step in range(max_steps):
        optimizer.zero_grad()
        pred = conv(inp)
        loss = nn.functional.mse_loss(pred, out)
        loss.backward(); optimizer.step(); scheduler.step()
        l = loss.item()
        if l < best_loss - 1e-7:
            best_loss = l; best_state = copy.deepcopy(conv.state_dict()); plateau = 0
        else:
            plateau += 1
        if step == 250 and best_loss > 0.3: return None
        if step == 600 and best_loss > 0.1: return None
        if plateau > 400 and best_loss > 0.01: return None
        if best_loss < 1e-6: break

    if best_loss > 0.005: return None
    conv.load_state_dict(best_state)
    weight = conv.weight.detach().numpy()

    # Prune near-zero weights
    weight_pruned = np.where(np.abs(weight) < 0.01, 0.0, weight)
    # Also try rounding to integers
    weight_rounded = np.round(weight)

    # Try pruned, rounded, original — pick smallest cost that works
    best_model = None
    best_cost = float('inf')
    for w_try in [weight_rounded, weight_pruned, weight]:
        m = make_conv_onnx(w_try, kernel_size=kernel_size)
        if check_model_correct(m, all_pairs):
            c = estimate_model_cost(m)
            if c < best_cost:
                best_cost = c; best_model = m
    return best_model

def try_two_layer_conv_fast(pairs, all_pairs, ks1=3, ks2=1, hidden=16, max_steps=2500, lr=0.015):
    try:
        import torch, torch.nn as nn
    except ImportError:
        return None
    if not all(shapes_match(p) for p in pairs): return None

    inp = torch.tensor(np.stack([grid_to_tensor(p["input"])[0] for p in pairs]), dtype=torch.float32)
    out = torch.tensor(np.stack([grid_to_tensor(p["output"])[0] for p in pairs]), dtype=torch.float32)

    pad1 = ks1 // 2; pad2 = ks2 // 2
    torch.manual_seed(0)
    model_nn = nn.Sequential(
        nn.Conv2d(C, hidden, kernel_size=ks1, padding=pad1, bias=True),
        nn.ReLU(),
        nn.Conv2d(hidden, C, kernel_size=ks2, padding=pad2, bias=False),
    )

    optimizer = torch.optim.Adam(model_nn.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_steps)
    best_loss = float("inf"); best_state = None; plateau = 0

    for step in range(max_steps):
        optimizer.zero_grad()
        pred = model_nn(inp)
        loss = nn.functional.mse_loss(pred, out)
        loss.backward(); optimizer.step(); scheduler.step()
        l = loss.item()
        if l < best_loss - 1e-7:
            best_loss = l; best_state = copy.deepcopy(model_nn.state_dict()); plateau = 0
        else:
            plateau += 1
        if step == 300 and best_loss > 0.5: return None
        if step == 800 and best_loss > 0.15: return None
        if plateau > 500 and best_loss > 0.005: return None
        if best_loss < 1e-6: break

    if best_loss > 0.005: return None
    model_nn.load_state_dict(best_state)
    w1 = model_nn[0].weight.detach().numpy()
    b1 = model_nn[0].bias.detach().numpy()
    w2 = model_nn[2].weight.detach().numpy()

    # Prune
    w1 = np.where(np.abs(w1) < 0.01, 0.0, w1)
    w2 = np.where(np.abs(w2) < 0.01, 0.0, w2)

    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    w1_t = onh.from_array(w1.astype(np.float32), name="w1")
    w1_n = oh.make_node("Constant", [], ["w1"], value=w1_t)
    b1_t = onh.from_array(b1.astype(np.float32), name="b1")
    b1_n = oh.make_node("Constant", [], ["b1"], value=b1_t)
    c1 = oh.make_node("Conv", ["input", "w1", "b1"], ["c1_out"],
                      kernel_shape=[ks1, ks1], pads=[pad1, pad1, pad1, pad1])
    relu = oh.make_node("Relu", ["c1_out"], ["relu"])
    w2_t = onh.from_array(w2.astype(np.float32), name="w2")
    w2_n = oh.make_node("Constant", [], ["w2"], value=w2_t)
    c2 = oh.make_node("Conv", ["relu", "w2"], ["output"],
                      kernel_shape=[ks2, ks2], pads=[pad2, pad2, pad2, pad2])
    nodes = [w1_n, b1_n, c1, relu, w2_n, c2]
    graph = oh.make_graph(nodes, "two_layer_conv", [X], [Y])
    m = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    m.ir_version = 8
    if check_model_correct(m, all_pairs):
        return m
    return None

# ============================================================
# SOLVER
# ============================================================
def solve_task(task_data, task_name="", time_budget=12.0):
    train_pairs = task_data.get("train", [])
    test_pairs = task_data.get("test", [])
    arc_gen = task_data.get("arc-gen", [])

    if not train_pairs:
        return make_identity_onnx()

    # Validate on much more arc-gen data than v2 (V2 used 10, now 50+)
    # Shuffle arc-gen but use seed for determinism
    rng = random.Random(42)
    ag = list(arc_gen)
    rng.shuffle(ag)
    all_pairs = train_pairs + test_pairs + ag[:MAX_ARC_GEN_VALIDATE]

    info = analyze_transformation(train_pairs)
    best_model = None; best_cost = float('inf')
    t_start = time.time()

    def try_model(model, label=""):
        nonlocal best_model, best_cost
        if model is None: return False
        if check_model_correct(model, all_pairs):
            cost = estimate_model_cost(model)
            if cost < best_cost:
                best_cost = cost; best_model = model
            return True
        return False

    def elapsed(): return time.time() - t_start

    # ---- CHEAP CHECKS ----
    if info.get("identity"):
        if try_model(make_identity_onnx(), "identity"):
            return best_model

    # Color maps (global) - prefer channel-gather when safe.
    # IMPORTANT: If a spatial transform also fits on training, the data is
    # ambiguous (values rotated could look like a color permutation). In that case
    # we must prefer the spatial transform which generalizes to arbitrary data.
    has_spatial_same_size = (
        (detect_rotation(train_pairs) == 180) or  # only rot180 preserves size
        (detect_flip(train_pairs) is not None) or
        detect_transpose(train_pairs) or
        detect_anti_transpose(train_pairs)
    )
    # Also check: for different-shape outputs, other spatial transforms might disguise
    has_spatial_any = (
        detect_rotation(train_pairs) is not None or
        detect_flip(train_pairs) is not None or
        detect_transpose(train_pairs) or
        detect_anti_transpose(train_pairs)
    )
    if info.get("color_map") and info["same_size"]:
        cmap = info["color_map"]
        if can_use_channel_gather_for_cmap(cmap, train_pairs) and not has_spatial_same_size:
            gi = build_channel_gather_indices(cmap)
            try_model(make_channel_gather_onnx(gi), "color_map_ch")
        w = build_color_map_weight(cmap)
        try_model(make_conv1x1_onnx(w), "color_map_conv")

    # Nonzero recolor (silhouette)
    tc = detect_nonzero_recolor(train_pairs)
    if tc is not None:
        try_model(make_conv1x1_onnx(build_nonzero_recolor_weight(tc)), "nonzero_recolor")

    # BG recolor
    tc = detect_bg_recolor(train_pairs)
    if tc is not None:
        try_model(make_conv1x1_onnx(build_bg_recolor_weight(tc)), "bg_recolor")

    # Color replace / swap - prefer channel-gather for bijective (safe) cases
    # but NOT when spatial transforms also fit (ambiguity).
    repl = detect_color_replace(train_pairs)
    if repl:
        cmap = {i: i for i in range(C)}; cmap.update(repl)
        if can_use_channel_gather_for_cmap(cmap, train_pairs) and not has_spatial_same_size:
            gi = build_channel_gather_indices(cmap)
            try_model(make_channel_gather_onnx(gi), "color_replace_ch")
        try_model(make_conv1x1_onnx(build_color_map_weight(cmap)), "color_replace_conv")

    sw = detect_color_swap(train_pairs)
    if sw and not has_spatial_same_size:
        # Swap is always bijective; with both-direction evidence from detect_color_swap
        # and no spatial transform, it's safe.
        a, b = sw
        cmap = {i: i for i in range(C)}; cmap[a] = b; cmap[b] = a
        gi = build_channel_gather_indices(cmap)
        try_model(make_channel_gather_onnx(gi), "color_swap_ch")
        try_model(make_conv1x1_onnx(build_color_map_weight(cmap)), "color_swap_conv")

    # Rotation (size-aware, maskless thanks to SAFE_PAD_IDX)
    rot = detect_rotation(train_pairs)
    if rot:
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            gi = build_rotation_gather(rot, ih, iw)
            try_model(make_gather_onnx(gi), f"rot_{rot}")

    flip = detect_flip(train_pairs)
    if flip:
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            try_model(make_gather_onnx(build_flip_gather(flip, ih, iw)), f"flip_{flip}")

    if detect_transpose(train_pairs):
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            try_model(make_gather_onnx(build_transpose_gather(ih, iw)), "transpose")
    if detect_anti_transpose(train_pairs):
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            try_model(make_gather_onnx(build_anti_transpose_gather(ih, iw)), "anti_transpose")

    # Crop / tile / scale
    crop = detect_crop(train_pairs)
    if crop:
        r0, c0, oh_, ow_ = crop
        try_model(make_gather_onnx(build_crop_gather(r0, c0, oh_, ow_)), "crop")

    tile = detect_tile(train_pairs)
    if tile:
        tr, tc = tile
        ih, iw = np.array(train_pairs[0]["input"]).shape
        oh_ = ih * tr; ow_ = iw * tc
        if oh_ <= H and ow_ <= W:
            try_model(make_gather_onnx(build_tile_gather(tr, tc, ih, iw)), "tile")

    sc = detect_scale(train_pairs)
    if sc:
        ih, iw = np.array(train_pairs[0]["input"]).shape
        oh_ = ih * sc; ow_ = iw * sc
        if oh_ <= H and ow_ <= W:
            try_model(make_gather_onnx(build_scale_gather(sc, ih, iw)), f"scale_{sc}")

    us = detect_upscale(train_pairs)
    if us:
        yf, xf = us
        ih, iw = np.array(train_pairs[0]["input"]).shape
        oh_ = ih * yf; ow_ = iw * xf
        if oh_ <= H and ow_ <= W:
            try_model(make_gather_onnx(build_nonuniform_scale_gather(yf, xf, ih, iw)),
                      f"upscale_{yf}x{xf}")

    # Pixel permutation (conservative)
    perm = detect_pixel_permutation(train_pairs)
    if perm is not None:
        try_model(make_gather_onnx(perm), "pixel_perm")

    # Mirror patterns
    if detect_mirror_h_concat(train_pairs):
        ih, iw = np.array(train_pairs[0]["input"]).shape
        try_model(make_gather_onnx(build_mirror_h_concat_gather(ih, iw)), "mirror_h")
    if detect_mirror_v_concat(train_pairs):
        ih, iw = np.array(train_pairs[0]["input"]).shape
        try_model(make_gather_onnx(build_mirror_v_concat_gather(ih, iw)), "mirror_v")

    if detect_self_concat_h(train_pairs):
        ih, iw = np.array(train_pairs[0]["input"]).shape
        try_model(make_gather_onnx(build_self_concat_h_gather(ih, iw)), "self_h")
    if detect_self_concat_v(train_pairs):
        ih, iw = np.array(train_pairs[0]["input"]).shape
        try_model(make_gather_onnx(build_self_concat_v_gather(ih, iw)), "self_v")

    qm = detect_quad_mirror(train_pairs)
    if qm:
        ih, iw = np.array(train_pairs[0]["input"]).shape
        try_model(make_gather_onnx(build_quad_mirror_gather(ih, iw, qm)), "quad_mirror")

    # Composite: spatial transform + color map.
    if rot and info.get("color_map"):
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            gi = build_rotation_gather(rot, ih, iw)
            cmap = info["color_map"]
            if can_use_channel_gather_for_cmap(cmap, train_pairs):
                ch_gi = build_channel_gather_indices(cmap)
                try_model(make_spatial_then_channel_gather_onnx(gi, ch_gi), "rot_ch_colormap")
            w = build_color_map_weight(cmap)
            try_model(make_gather_then_conv1x1_onnx(gi, w), "rot_colormap")
    if flip and info.get("color_map"):
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            gi = build_flip_gather(flip, ih, iw)
            cmap = info["color_map"]
            if can_use_channel_gather_for_cmap(cmap, train_pairs):
                ch_gi = build_channel_gather_indices(cmap)
                try_model(make_spatial_then_channel_gather_onnx(gi, ch_gi), "flip_ch_colormap")
            w = build_color_map_weight(cmap)
            try_model(make_gather_then_conv1x1_onnx(gi, w), "flip_colormap")

    # Grid-of-grids
    gog = detect_grid_of_grids_fixed(train_pairs)
    if gog:
        br, bc, oh_, ow_ = gog
        r0 = br * oh_; c0 = bc * ow_
        try_model(make_gather_onnx(build_crop_gather(r0, c0, oh_, ow_)), "grid_select")

    # Symmetry completion
    if info["same_size"] and info["all_same_in_size"]:
        ih, iw = np.array(train_pairs[0]["input"]).shape
        if detect_h_symmetry_complete(train_pairs):
            try_model(make_symmetry_completion_onnx(build_h_sym_complete_gathers(ih, iw)), "h_sym")
        if detect_v_symmetry_complete(train_pairs):
            try_model(make_symmetry_completion_onnx(build_v_sym_complete_gathers(ih, iw)), "v_sym")
        if detect_both_symmetry_complete(train_pairs):
            try_model(make_symmetry_completion_onnx(build_both_sym_complete_gathers(ih, iw)), "both_sym")
        if ih == iw:
            if detect_diag_symmetry_complete(train_pairs):
                g = build_diag_sym_complete_gathers(ih, iw)
                if g is not None:
                    try_model(make_symmetry_completion_onnx(g), "diag_sym")
            if detect_all4_symmetry_complete(train_pairs):
                g = build_all4_sym_complete_gathers(ih, iw)
                if g is not None:
                    try_model(make_symmetry_completion_onnx(g), "all4_sym")

    if best_model is not None:
        return best_model

    # ---- LEARNED CONV ----
    for ks in [1, 3]:
        if elapsed() > time_budget: break
        m = try_learned_conv_fast(train_pairs, all_pairs, kernel_size=ks)
        if try_model(m, f"lconv_{ks}"):
            if best_cost < 10000:
                return best_model

    if best_model is not None:
        return best_model

    # Two-layer conv
    for ks1, ks2, hidden in [(3, 1, 16), (3, 1, 24), (3, 3, 16), (5, 1, 12)]:
        if elapsed() > time_budget: break
        m = try_two_layer_conv_fast(train_pairs, all_pairs, ks1=ks1, ks2=ks2, hidden=hidden)
        if try_model(m, f"lconv2_{ks1}_{ks2}_{hidden}"):
            break

    if best_model is not None:
        return best_model

    # Constant output
    if info.get("const_output"):
        c_out = grid_to_tensor(train_pairs[0]["output"])
        try_model(make_const_onnx(c_out), "const")

    return best_model

# ============================================================
# I/O
# ============================================================
def save_model(model, path):
    with open(path, "wb") as f:
        f.write(model.SerializeToString())

def find_task_files():
    for d in [TASK_DIR, Path("/kaggle/input"), Path(".")]:
        if d.exists():
            files = list(d.glob("task*.json"))
            if files: return sorted(files)
    task_files = []
    root = "/kaggle/input" if Path("/kaggle/input").exists() else "."
    for r, _, files in os.walk(root):
        for f in files:
            if f.startswith("task") and f.endswith(".json"):
                task_files.append(Path(r) / f)
    return sorted(task_files)

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    task_files = find_task_files()
    if not task_files:
        print("ERROR: no task files"); sys.exit(1)
    print(f"Found {len(task_files)} task files")

    solved = []; failed = []; onnx_files = []
    total_score = 0.0
    start = time.time()

    for task_path in task_files:
        task_name = task_path.stem
        task_num = task_name.replace("task", "")
        try:
            task_data = load_task(task_path)
        except Exception as e:
            print(f"{task_name}: load error {e}")
            failed.append(task_name); continue

        t0 = time.time()
        print(f"Processing {task_name}...", end=" ", flush=True)
        try:
            model = solve_task(task_data, task_name, time_budget=10.0)
        except Exception as e:
            print(f"ERROR: {e}")
            failed.append(task_name); continue
        dt = time.time() - t0

        if model is None:
            print(f"No solution ({dt:.1f}s)")
            failed.append(task_name); continue

        out_path = OUTPUT_DIR / f"task{task_num}.onnx"
        try:
            save_model(model, out_path)
            onnx_files.append(out_path)
            cost = estimate_model_cost(model)
            score = max(1, 25 - math.log(max(1, cost)))
            total_score += score
            print(f"Solved cost={cost} score={score:.1f} ({dt:.1f}s)")
            solved.append(task_name)
        except Exception as e:
            print(f"Save error: {e}")
            failed.append(task_name)

    total = time.time() - start
    print(f"\n{'='*60}")
    print(f"Solved: {len(solved)}/{len(task_files)}  Failed: {len(failed)}")
    print(f"Total local estimated score: {total_score:.1f}  Total time: {total:.1f}s")

    if onnx_files:
        zip_path = OUTPUT_DIR / "submission.zip"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for f in onnx_files:
                zf.write(f, f.name)
        print(f"Submission: {zip_path} ({zip_path.stat().st_size/1024:.1f} KB)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 77.1 MB/s eta 0:00:00
Found 400 task files
Processing task001... No solution (3.2s)
Processing task002... No solution (15.7s)
Processing task003... No solution (0.0s)
Processing task004... No solution (12.2s)
Processing task005... No solution (14.1s)
Processing task006... No solution (0.0s)
Processing task007... No solution (13.9s)
Processing task008... No solution (14.0s)
Processing task009... No solution (13.4s)
Processing task010... No solution (12.5s)
Processing task011... No solution (15.2s)
Processing task012... No solution (12.2s)
Processing task013... No solution (13.9s)
Processing task014... No solution (0.0s)
Processing task015... Solved cost=814500 score=11.4 (1.7s)
Processing task016... Solved cost=50 score=21.1 (0.0s)
Processing task017... No solution (16.5s)
Processing task018... No solution (13.6s)
Processing task019... No solution (0.0s)
Processing task020... No solution (13.5s)
Processing task021... No solution (